# 1. Import des données

In [1]:
import pandas as pd
import numpy as np

stock_prices = pd.read_excel("case study - Individual_stock_data.xlsx", sheet_name=0)
stock_prices["Unnamed: 0"] = pd.to_datetime(stock_prices["Unnamed: 0"], format="%Y-%m-%d")
stock_prices.set_index("Unnamed: 0", inplace=True)
stock_prices.index.rename("Date", inplace=True)
stock_prices = stock_prices.resample("W").ffill()

carbon_data = pd.read_excel("case study - Individual_stock_data.xlsx", sheet_name=1)
carbon_data.rename(columns={"Unnamed: 0": "Code"}, inplace=True)

sp500 = pd.read_excel("case study - Individual_stock_data.xlsx", sheet_name=2)
sp500["Name"] = pd.to_datetime(sp500["Name"], format="%Y-%m-%d")
sp500.set_index("Name", inplace=True)
sp500.index.rename("Date", inplace=True)
sp500 = sp500.resample("W").ffill()


# 2. Traitement

On se limite aux entreprises dont on dispose des données carbone et d'un historique de données complet depuis 2012.

In [ ]:
carbon_data = carbon_data.dropna(subset=["SCOPE1", "SCOPE2"])
carbon_data["WEIGHT_IN_SP500"] = carbon_data["MARKET_VALUE"]/carbon_data["MARKET_VALUE"].sum()
carbon_data = carbon_data.sort_values("INTENSITY", axis=0, ascending=True).dropna()
carbon_data.set_index("Code", inplace=True)

stock_prices = stock_prices[stock_prices["2012":"2020"].dropna(axis=1).columns]
carbon_data = carbon_footprint[carbon_data.index.isin(stock_prices.columns)]

In [ ]:
by_sector = carbon_footprint[["SECTOR_NAME", "SCOPE1", "SCOPE2", "EMISSIONS", "MARKET_VALUE", "REVENUES"]].groupby(["SECTOR_NAME"]).sum()

In [ ]:
by_sector["WEIGHT_IN_SP500"] = by_sector["MARKET_VALUE"]/by_sector["MARKET_VALUE"].sum() 
by_sector

# 3. Sélection des 50 titres

In [ ]:
import pyomo.environ as pyo

alpha = 0.5
lamb = 100
CI_sp500 = (carbon_footprint["INTENSITY"]*carbon_footprint["WEIGHT_IN_SP500"]).sum()

m = pyo.ConcreteModel(name="selection 50 in SP500")

m.Stocks = pyo.Set(initialize=carbon_footprint.index.to_list())
m.Sectors = pyo.Set(initialize=by_sector.index.to_list())

m.CI = pyo.Param(m.Stocks, initialize=carbon_footprint["INTENSITY"].to_dict())
m.b = pyo.Param(m.Stocks, initialize=carbon_footprint["WEIGHT_IN_SP500"].to_dict())
m.Bs = pyo.Param(m.Sectors, initialize=by_sector["WEIGHT_IN_SP500"].to_dict())
m.CI_sp500 = pyo.Param(initialize=float(CI_sp500))
m.alpha = pyo.Param(initialize=float(alpha))
m.lamb = pyo.Param(initialize=float(lamb))

sector = carbon_footprint["SECTOR_NAME"].to_dict()
m.I_by_S = pyo.Set(
    m.Sectors,
    initialize=lambda mm, s: [i for i in mm.Stocks if sector[i] == s],
    ordered=False
)


# VARIABLES
m.z  = pyo.Var(m.Stocks, within=pyo.Binary)               # 1 si titre sélectionné
m.W  = pyo.Var(within=pyo.NonNegativeReals)          # total benchmark weight covered
m.Ws = pyo.Var(m.Sectors, within=pyo.NonNegativeReals)     # sector benchmark weight covered
m.d  = pyo.Var(m.Sectors, within=pyo.NonNegativeReals)     # abs deviation from sector mix

# CONSTRAINTS

# (1) select exactly 50 names
m.card = pyo.Constraint(expr=sum(m.z[i] for i in m.Stocks) == 50)

# (2) total covered benchmark weight
m.defW = pyo.Constraint(expr=m.W == sum(m.b[i] * m.z[i] for i in m.Stocks))

# (2b) covered benchmark weight per sector
def defWs(mm, s):
    return mm.Ws[s] == sum(mm.b[i] * mm.z[i] for i in mm.I_by_S[s])
    
m.defWs = pyo.Constraint(m.Sectors, rule=defWs)

# (3) sector-mix deviation: d_s >= |Ws[s] - Bs[s]*W|
def _dev_pos(mm, s):
    return mm.d[s] >= mm.Ws[s] - mm.Bs[s] * mm.W
def _dev_neg(mm, s):
    return mm.d[s] >= -(mm.Ws[s] - mm.Bs[s] * mm.W)
    
m.dev_pos = pyo.Constraint(m.Sectors, rule=_dev_pos)
m.dev_neg = pyo.Constraint(m.Sectors, rule=_dev_neg)

# (5) carbon intensity constraint on the proxy cap-weight portfolio:
#     sum_i b_i CI_i z_i <= alpha * CIsp * W
m.carbon = pyo.Constraint(
    expr=sum(m.b[i] * m.CI[i] * m.z[i] for i in m.Stocks) <= m.alpha * m.CI_sp500 * m.W
)

# --- Objective: maximize coverage - lambda * sector deviation
m.obj = pyo.Objective(
    expr=m.W - m.lamb * sum(m.d[s] for s in m.Sectors),
    sense=pyo.maximize
)


optimizer = pyo.SolverFactory("gurobi")
optimizer.options["TimeLimit"] = 60
optimizer.solve(m)



In [ ]:
list_selec = [stock for stock in carbon_footprint.index if pyo.value(m.z[stock]) == 1]
selec = carbon_footprint[carbon_footprint.index.isin(list_selec)].copy()
selec["WEIGHT_IN_SELEC"] = selec["MARKET_VALUE"]/selec["MARKET_VALUE"].sum()

In [ ]:
CI_selec = (selec["INTENSITY"]*selec["WEIGHT_IN_SELEC"]).sum()
print(f"Carbon intensity of S&P500 : {round(CI_sp500, 1)} \t Carbon intensity of selection : {round(CI_selec,1)} \t ratio = {round(CI_selec/CI_sp500, 3)}")

In [ ]:
print(f"Weight of S&P500 covered in selection of 50 stocks : {round(pyo.value(m.W)*100,2)} %")

In [ ]:
sector_selection = selec[["SECTOR_NAME", "SCOPE1", "SCOPE2", "EMISSIONS", "MARKET_VALUE", "REVENUES"]].groupby(["SECTOR_NAME"]).sum()
sector_selection["WEIGHT_IN_SELECTION"] = sector_selection["MARKET_VALUE"]/sector_selection["MARKET_VALUE"].sum() 
sector_selection["WEIGHT_IN_SP500"] = by_sector["WEIGHT_IN_SP500"]

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots()
(sector_selection[["WEIGHT_IN_SELECTION", "WEIGHT_IN_SP500"]]*100).plot.bar(ax=ax)
ax.set_ylabel("%")
ax.set_xlabel("Sector name")
ax.legend()
plt.plot()
plt.savefig("comparison_weights_sector.png",bbox_inches='tight')

In [ ]:
selec.groupby("SECTOR_NAME")["NAME"].apply(list).to_excel("selection.xlsx")

# 4. Constitution des différents portefeuilles

In [ ]:
sp500_returns = sp500.pct_change()
stock_prices = stock_prices[list_selec]
stock_returns = stock_prices.pct_change()["2012":"2020"]
carbon_footprint_selec = carbon_footprint.loc[selec.index]

In [ ]:
Cov_i_SP500 = np.array(stock_returns.apply(lambda s: s.cov(sp500_returns["S&P 500 COMPOSITE - TOT RETURN IND"])))

In [ ]:

CI_sp500 = (carbon_footprint["INTENSITY"]*carbon_footprint["WEIGHT_IN_SP500"]).sum()

def compute_weights(returns, strategy, sp500_returns, carbon_footprint=carbon_footprint_selec, CI_sp500=CI_sp500, alpha=0.5):
    
    m = pyo.ConcreteModel(name="portfolio optimization")

    # Sets
    m.N = pyo.Set(initialize=list(returns.columns), ordered=True)
    
    #Parameters
    n_stocks = len(returns.columns)
    m.n_stocks = pyo.Param(initialize=n_stocks)
    m.CI = pyo.Param(m.N, initialize=carbon_footprint["INTENSITY"].to_dict())
    m.CI_sp500 = pyo.Param(initialize=CI_sp500)
    m.alpha = pyo.Param(initialize=alpha)

    #Variables
    m.w = pyo.Var(m.N, within=pyo.NonNegativeReals)

    #Constraints
    m.sum_weights = pyo.Constraint(rule=lambda m: sum(m.w[i] for i in m.N) == 1)
    m.carbon = pyo.Constraint(expr=sum(m.w[i] * m.CI[i] for i in m.N) <= m.alpha * m.CI_sp500)
  
    #Objective function : depends on portfolio type
    
    match strategy:
        case "GMV":
            Sigma = returns.cov()
            m.Sigma = pyo.Param(m.N, m.N, initialize=Sigma.stack().to_dict())
            def GMV(m):
                return sum(m.w[i]*m.w[j]*m.Sigma[i, j] for i in m.N for j in m.N)

            m.obj = pyo.Objective(rule=GMV, sense=pyo.minimize)
                
        case "Max Diversification":
            Sigma = returns.cov()
            sigma = returns.std()
            m.Sigma = pyo.Param(m.N, m.N, initialize=Sigma.stack().to_dict())
            m.sigma = pyo.Param(m.N, initialize=sigma.to_dict())
            
            def MaxDiv(m):
                return sum(m.w[i]*m.sigma[i] for i in m.N)
                 # Gurobi-friendly (QCP) : maximise sum(w_i*sigma_i) s.c. variance <= 1
                # puis renormalisation pour sommer à 1
            m.del_component(m.sum_weights)
            m.del_component(m.carbon)
    
            m.var_cap = pyo.Constraint(expr=sum(m.w[i] * m.w[j] * m.Sigma[i, j] for i in m.N for j in m.N) <= 1.0)
            m.carbon = pyo.Constraint(expr=sum(m.w[i] * m.CI[i] for i in m.N) <= m.alpha * m.CI_sp500 * sum(m.w[i] for i in m.N))
            m.obj = pyo.Objective(rule=MaxDiv,sense=pyo.maximize)
        
            
        case "Max Decorrelation":
            Corr = returns.corr()
            m.Corr = pyo.Param(m.N, m.N, initialize=Corr.stack().to_dict())
            def MaxDecorrel(m):
                return sum(m.w[i] * m.w[j] * m.Corr[i, j] for i in m.N for j in m.N)

            m.obj = pyo.Objective(rule=MaxDecorrel, sense=pyo.minimize)
  
        case "max_ENC_dollar":
            def max_ENC_dollar(m):
                return sum(m.w[i]**2 for i in m.N)

            m.obj = pyo.Objective(rule=max_ENC_dollar, sense=pyo.minimize)

      
        case "Min Tracking Error":
            Sigma = returns.cov()
            Cov_i_SP500 = returns.apply(lambda s: s.cov(sp500_returns["S&P 500 COMPOSITE - TOT RETURN IND"]))
            
            m.Sigma = pyo.Param(m.N, m.N, initialize=Sigma.stack().to_dict())
            m.Cov_i_SP500 = pyo.Param(m.N, initialize=Cov_i_SP500.to_dict())
              
            def MinTE(m):
                return sum(m.w[i]*m.w[j]*m.Sigma[i, j] for i in m.N for j in m.N) - 2*sum(m.w[i]*m.Cov_i_SP500[i] for i in m.N)   # TE = Var(R_p) + Var_(R_SP500) - 2 Cov(R_p, R_SP500) -> Var(R_SP500) is a constant, so no need for it in the objective fucntion

            m.obj = pyo.Objective(rule=MinTE, sense=pyo.minimize)

    optimizer = pyo.SolverFactory("gurobi") 
    optimizer.options["TimeLimit"] = 60
    optimizer.solve(m)
    
    w = {i: pyo.value(m.w[i]) for i in list(m.N)}
    w = pd.Series(w)

    if strategy == "Max Diversification":
        # Renormalisation uniquement pour MaxDiv version QCP
        s = w.sum()
        if s > 0:
            w = w / s

    return w   

def backtest_rolling(returns, strategy, sp500_returns=sp500_returns, window=2):
    T = returns.index.get_loc(returns.index.max())
    t0 = window*52+1
    portfolio = {}

    R_OOS = []
    dates_OOS = []

    for t in range(t0, T): # need at least t0 data points to begin the calculation
        R_insample = returns.iloc[t-window*52:t].copy() 
        R_sp500 = sp500_returns.loc[R_insample.index].copy()
       
        if t%13 == 0 or t==t0: #quarterly rebalancing
            w = compute_weights(R_insample, strategy, R_sp500)
            portfolio[returns.index[t]] = w

        R_OOS.append(w.T @ returns.iloc[t])
        dates_OOS.append(returns.index[t])
        

    return pd.Series(R_OOS, index=dates_OOS, name=strategy), pd.DataFrame.from_dict(portfolio, orient="index")

strategies = ["GMV", "Max Diversification", "Max Decorrelation", "max_ENC_dollar", "Min Tracking Error"]
R_OOS = {}
portfolios = {}

for strat in strategies:
    print(strat)
    R_OOS[strat], portfolios[strat] = backtest_rolling(stock_returns, strat)

R_OOS = pd.DataFrame(R_OOS)
print("Done!")


In [ ]:
sector_map = carbon_footprint["SECTOR_NAME"]
portfolios_sector = {}
for strat, portfolio in portfolios.items():
    sector_map = sector_map.reindex(portfolio.columns)
    sector_weights = portfolio.T.groupby(sector_map).sum().T 
    portfolios_sector[strat] = sector_weights.add_prefix("SECTOR_")

In [ ]:
import matplotlib.pyplot as plt

R_OOS["SP500"] = sp500["S&P 500 COMPOSITE - TOT RETURN IND"].pct_change()["2014":"2020"]


fig, ax = plt.subplots()

for strat, r in R_OOS.items():
    curve = (1 + r).cumprod()
    if strat=="SP500":
        ax.plot(curve.index, curve.values, label=f"{strat}",  linestyle=":", color="black")
    else:
        ax.plot(curve.index, curve.values, label=f"{strat}")  # index = DateTimeIndex
    


ax.set_ylabel("Cumulative returns")
ax.set_xlabel("Date")

plt.legend()
plt.gcf().autofmt_xdate()  
plt.savefig("returns_portfolios.png")
plt.show()


In [ ]:
df = portfolios_sector["Max Diversification"]

df.index = df.index.date

ax = df.plot.area(figsize=(12,5), linewidth=0, cmap="turbo")
ax.set_ylabel("Weight")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, 1.27), ncol=4, frameon=False)
plt.tight_layout()
plt.savefig("sector_weights_mdiv.png")
plt.show()



In [ ]:
import seaborn as sns

sns.heatmap(df.transpose())

In [ ]:
annualized_returns = {}
annualized_volatility = {}
sharpe_ratios = {}
TE_SP500 = {}

for strat in strategies + ["SP500"]:
    annualized_returns[strat] = ( (1+R_OOS[strat]).prod() ** (52/R_OOS[strat].shape[0]) - 1) * 100
    annualized_volatility[strat] = R_OOS[strat].std() * np.sqrt(52) * 100
    sharpe_ratios[strat] = ( R_OOS[strat].mean()/R_OOS[strat].std() ) * np.sqrt(52)
    TE_SP500[strat] = (R_OOS[strat]-R_OOS["SP500"]).std() * np.sqrt(52) * 100
    

for strat in strategies:
    ENC[strat] = 1 / (portfolios[strat].pow(2).sum(axis=1))

metrics = pd.DataFrame()
metrics["Annualized returns"] = annualized_returns
metrics["Annualized volatility"] = annualized_volatility
metrics["Sharpe ratio"] = sharpe_ratios
metrics["Tracking error vs. SP500"] = TE_SP500
metrics["ENC"] = ENC
metrics.round(2)

In [ ]:
ENC = 1 / (portfolios["Max Diversification"].pow(2).sum(axis=1))
ENC.mean()

In [ ]:
metrics.index = ["GMV", "MDiv", "MDecorr", "ENC_$", "min_TE", "SP500"]
fig, axs = plt.subplots(2,2, layout="constrained")

metrics["Annualized returns"].plot.bar(label="Annualized returns", ax=axs[0,0])
axs[0,0].legend(loc="lower center")
axs[0,0].set_ylabel("%")

metrics["Annualized volatility"].plot.bar(label="Annualized volatility", ax=axs[1, 0], color="purple")
axs[1,0].legend(loc="lower center")
axs[1,0].set_ylabel("%")

metrics["Sharpe ratio"].plot.bar(label="Sharpe ratio", ax=axs[0, 1], color="green")
axs[0,1].legend(loc="lower center")


metrics["Tracking error vs. SP500"].plot.bar(label="Annualized tracking error", ax=axs[1, 1], color="orange")
axs[1,1].legend(loc="lower center")
axs[1,1].set_ylabel("%")

plt.savefig("results_optim.jpg")
plt.show()
